In [3]:
import json
from pathlib import Path
import pandas as pd

# Data folder path check cheyyuka
DATA_DIR = Path("data") if Path("data").exists() else Path(".")

# Munpu save cheytha priority results or clean dataset load cheyyuka
if (DATA_DIR / "tickets_with_priority.csv").exists():
    df = pd.read_csv(DATA_DIR / "tickets_with_priority.csv")
elif Path("ticket_priority_results.csv").exists():
    df = pd.read_csv("ticket_priority_results.csv")
    # Category column illenkil tickets_clean.csv-il irunthu merge cheyyuka
    if "category" not in df.columns and (DATA_DIR / "tickets_clean.csv").exists():
        clean_df = pd.read_csv(DATA_DIR / "tickets_clean.csv")
        df = pd.merge(df, clean_df[["ticket_id", "category"]], on="ticket_id", how="left")
else:
    df = pd.read_csv(DATA_DIR / "tickets_clean.csv")

print(f"Loaded {len(df)} tickets for response templates.")

# Predefined response templates based on ticket categories
RESPONSE_TEMPLATES = {
    "Billing and Payments": (
        "Hello,\n\nThank you for reaching out regarding your billing inquiry. "
        "We are currently reviewing your transaction details and will resolve any discrepancies within 24 hours.\n\n"
        "Best regards,\nCustomer Support Team"
    ),
    "Technical Issues": (
        "Hello,\n\nWe apologize for the technical difficulties you are experiencing. "
        "Our engineering team has been notified and is actively investigating the issue.\n\n"
        "Best regards,\nTechnical Support Team"
    ),
    "Account Access": (
        "Hello,\n\nWe understand you are facing trouble accessing your account. "
        "Please use the password reset link sent to your registered email, or let us know if you need further assistance with two-factor authentication.\n\n"
        "Best regards,\nSecurity Support Team"
    ),
    "Refund Requests": (
        "Hello,\n\nThank you for your request. We have initiated the review process for your refund. "
        "Approved refunds typically reflect in your account within 3 to 5 business days.\n\n"
        "Best regards,\nBilling & Finance Team"
    ),
    "Product or Service Complaints": (
        "Hello,\n\nWe sincerely apologize for the unsatisfactory experience. "
        "We value your feedback and are looking into this matter to ensure it does not happen again.\n\n"
        "Best regards,\nCustomer Relations Team"
    )
}

def generate_suggested_response(category):
    """Generate a draft response based on the ticket category."""
    # Category column checking
    cat = str(category) if pd.notna(category) else "General"
    return RESPONSE_TEMPLATES.get(cat, "Hello, thank you for contacting support. We are looking into your issue.")

# All tickets suggested response add 
if "category" in df.columns:
    df["suggested_response"] = df["category"].apply(generate_suggested_response)
else:
    # Category column illenkil oru default template kodukkuka
    df["suggested_response"] = "Hello, thank you for contacting support. We are looking into your issue."

# Sample ticket and response display 
print("\nSample Ticket & Suggested Response:")
sample = df.iloc[0]
print(f"Message: {sample.get('complaint', sample.get('text', 'N/A'))}")
print(f"Suggested Response:\n{sample['suggested_response']}")

# Result new CSV file to save 
output_file = "ticket_responses_results.csv"
df.to_csv(output_file, index=False)
print(f"\nSaved tickets with suggested responses to: {output_file}")

Loaded 8 tickets for response templates.

Sample Ticket & Suggested Response:
Message: I was charged twice and need help immediately
Suggested Response:
Hello,

Thank you for reaching out regarding your billing inquiry. We are currently reviewing your transaction details and will resolve any discrepancies within 24 hours.

Best regards,
Customer Support Team

Saved tickets with suggested responses to: ticket_responses_results.csv
